# 03 — Feature Engineering
**Owner: Parul** | Santander Customer Satisfaction

This notebook covers all feature engineering steps:
- Rule-based flag features (is_elderly, saldo_zero, var38_is_mode, saldo5_ult3_zero)
- One-hot encoding of var36
- Feature type documentation
- Select top 100 features by correlation

## Step 1 — Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# load the cleaned dataset saved from cleaning pipeline
df = pd.read_csv('../data/processed/clean_train.csv')

# separate features and target
y = df['TARGET']
X = df.drop(columns=['TARGET'])

print("Shape:", X.shape)
print("Features:", X.shape[1])

## Step 2 — Flag: is_elderly (var15 > 80)

In [ ]:
# flag elderly customers (age > 80)
# from Day 2 analysis: very few elderly customers are unhappy
X['is_elderly'] = (X['var15'] > 80).astype(int)

print(f"is_elderly: {X['is_elderly'].sum()} customers flagged")

# test AUC lift
auc = roc_auc_score(y, X['is_elderly'])
print(f"is_elderly AUC: {auc:.4f}")

## Step 3 — Flag: saldo_zero (all balances = 0)

In [ ]:
# find all saldo (balance) columns
saldo_cols = [col for col in X.columns if 'saldo' in col]
print(f"Saldo columns found: {len(saldo_cols)}")

# flag customers where ALL balance columns are zero (likely inactive)
X['saldo_zero'] = (X[saldo_cols] == 0).all(axis=1).astype(int)

print(f"saldo_zero: {X['saldo_zero'].sum()} customers flagged")

# test AUC lift
auc = roc_auc_score(y, X['saldo_zero'])
print(f"saldo_zero AUC: {auc:.4f}")

## Step 4 — Flag: var38_is_mode (sentinel value)

In [ ]:
# var38 = likely mortgage/income value
# 117310.979 is a sentinel value meaning data is missing
train_raw = pd.read_csv('../data/raw/train.csv')
var38_mode = train_raw['var38'].mode()[0]
print(f"var38 sentinel value: {var38_mode}")
print(f"Customers with sentinel: {(train_raw['var38'] == var38_mode).sum()}")

# flag customers where var38 equals the sentinel
X['var38_is_mode'] = (train_raw['var38'] == var38_mode).astype(int)

print(f"var38_is_mode: {X['var38_is_mode'].sum()} customers flagged")

# test AUC lift
auc = roc_auc_score(y, X['var38_is_mode'])
print(f"var38_is_mode AUC: {auc:.4f}")

## Step 5 — Flag: saldo5_ult3_zero (3-month avg balance = 0)

In [ ]:
# flag customers where 3-month average balance for product 5 is zero
X['saldo5_ult3_zero'] = (train_raw['saldo_medio_var5_ult3'] == 0).astype(int)

print(f"saldo5_ult3_zero: {X['saldo5_ult3_zero'].sum()} customers flagged")

# test AUC lift
auc = roc_auc_score(y, X['saldo5_ult3_zero'])
print(f"saldo5_ult3_zero AUC: {auc:.4f}")

## Step 6 — One-Hot Encode var36

In [ ]:
# check unique values of var36
print("Unique values in var36:", sorted(X['var36'].unique()))
print("Value counts:")
print(X['var36'].value_counts().sort_index())

# one-hot encode var36
var36_dummies = pd.get_dummies(X['var36'], prefix='var36')
print("\nNew columns created:", var36_dummies.columns.tolist())

# add new columns and drop original var36
X = pd.concat([X, var36_dummies], axis=1)
X = X.drop(columns=['var36'])

print("Shape after one-hot encoding:", X.shape)

## Step 7 — Feature Type Documentation

In [ ]:
# categorical features (one-hot encoded)
categorical_cols = [col for col in X.columns if col.startswith('var36_')]

# engineered flag features (created by Parul)
flag_cols = ['is_elderly', 'saldo_zero', 'var38_is_mode', 'saldo5_ult3_zero']

# numeric features (everything else)
numeric_cols = [col for col in X.columns if col not in categorical_cols and col not in flag_cols]

print("===== FEATURE TYPE SUMMARY =====")
print(f"Categorical (one-hot) : {len(categorical_cols)} → {categorical_cols}")
print(f"Engineered flags      : {len(flag_cols)} → {flag_cols}")
print(f"Numeric features      : {len(numeric_cols)}")
print(f"Total                 : {X.shape[1]}")

## Step 8 — Engineered Feature AUC Summary

In [ ]:
# summary of all engineered features and their AUC lift
engineered = ['is_elderly', 'saldo_zero', 'var38_is_mode', 'saldo5_ult3_zero']

print("===== ENGINEERED FEATURE AUC SUMMARY =====")
print(f"{'Feature':<20} {'AUC':>8} {'Useful?':>10}")
print("-" * 42)
for col in engineered:
    auc = roc_auc_score(y, X[col])
    useful = '✅ YES' if auc > 0.55 else '❌ Weak'
    print(f"{col:<20} {auc:>8.4f} {useful:>10}")

## Step 9 — Save Final Feature Engineering Output

In [ ]:
# add TARGET back and save
X['TARGET'] = y.values
X.to_csv('../data/processed/feature_engineered_train.csv', index=False)

print(f"Saved! Final shape: {X.shape}")
print(f"File: ../data/processed/feature_engineered_train.csv")